# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vasavijoshi/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule idea

I will test two signals before encoding the baseline rule:

1. **Staleness (`days_since_last_update`)** — older pages should be stronger refresh candidates because this signal is linked to FlyRank's refresh/staleness flags.
2. **Search volume (`impressions_90d`)** — pages with meaningful search visibility may have more value to protect or improve.

The final rule will use only signals available in the current dataset and will produce a score, one reason code, and an action label.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd

# Load the starter dataset
if not os.path.exists("/content/flyrank-ml-internship"):
    !git clone https://github.com/Vasavijoshi/flyrank-ml-internship.git
else:
    print("Repository already cloned.")

df = pd.read_csv(
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

print("\nAll columns:")
for col in df.columns:
    print("-", col)

print("\nPotential staleness / date columns:")
print([
    col for col in df.columns
    if any(term in col.lower() for term in ["date", "update", "stale", "age"])
])

print("\nPotential volume columns:")
print([
    col for col in df.columns
    if any(term in col.lower() for term in ["impression", "volume"])
])

Repository already cloned.
Rows: 30,000
Columns: 44

All columns:
- content_id
- client_id
- search_volume
- competition
- competition_level
- cpc
- content_type
- main_intent
- word_count
- char_count
- provider_used
- model_used
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- days_with_impressions
- days_with_sessions
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- content_age_days
- age_tier
- age_tier_order
- days_since_last_update
- freshness_tier
- word_count_tier
- char_count_tier
- ctr
- avg_position
- engagement_rate
- scroll_rate
- ai_traffic_pct
- impression_tier
- position_tier
- trend_direction
- trend_pct

Potential staleness / date columns:
['pageviews_90d', 'engaged_sessions_90d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'engagement_rate']

Potential volume columns:
['searc

In [2]:
# Signal check 1: staleness

staleness_check = (
    df.assign(
        staleness_bucket=pd.cut(
            df["days_since_last_update"],
            bins=[-float("inf"), 90, 180, 365, float("inf")],
            labels=[
                "<=90 days",
                "91-180 days",
                "181-365 days",
                ">365 days"
            ]
        )
    )
    .groupby("staleness_bucket", observed=False)
    .size()
    .reset_index(name="n")
)

staleness_check["pct"] = (
    staleness_check["n"] / len(df) * 100
).round(1)

print("Staleness signal bucket table:")
display(staleness_check)

print(
    "\nVerdict: CONFIRMED — days_since_last_update provides "
    "a directly measurable staleness signal, including the 180-day "
    "refresh threshold used by the baseline logic."
)

Staleness signal bucket table:


,staleness_bucket,n,pct
0,<=90 days,20655,68.8
1,91-180 days,9171,30.6
2,181-365 days,169,0.6
3,>365 days,5,0.0



Verdict: CONFIRMED — days_since_last_update provides a directly measurable staleness signal, including the 180-day refresh threshold used by the baseline logic.


In [3]:
# Signal check 2: search visibility / volume

volume_check = (
    df.assign(
        volume_bucket=pd.cut(
            df["impressions_90d"],
            bins=[-float("inf"), 100, 250, 500, float("inf")],
            labels=[
                "<100 impressions",
                "100-249 impressions",
                "250-499 impressions",
                ">=500 impressions"
            ]
        )
    )
    .groupby("volume_bucket", observed=False)
    .size()
    .reset_index(name="n")
)

volume_check["pct"] = (
    volume_check["n"] / len(df) * 100
).round(1)

print("Search-volume signal bucket table:")
display(volume_check)

print(
    "\nVerdict: CONFIRMED — impressions_90d provides a measurable "
    "search-visibility signal, including the 500-impression threshold "
    "used by FlyRank's volume-based quick-win logic."
)

Search-volume signal bucket table:


,volume_bucket,n,pct
0,<100 impressions,8006,26.7
1,100-249 impressions,2622,8.7
2,250-499 impressions,2657,8.9
3,>=500 impressions,16715,55.7



Verdict: CONFIRMED — impressions_90d provides a measurable search-visibility signal, including the 500-impression threshold used by FlyRank's volume-based quick-win logic.


### Baseline rule

The baseline score combines two confirmed signals: staleness and search visibility.

- Staleness contributes 0–2 points: 0 for <=90 days, 1 for 91–180 days, and 2 for >180 days.
- Search visibility contributes 0–2 points: 0 for <250 impressions, 1 for 250–499 impressions, and 2 for >=500 impressions over 90 days.
- Pages scoring 3 or 4 are labelled **Refresh** because they are both relatively stale and meaningfully visible in search.
- Pages scoring below 3 are labelled **Monitor**.

The reason code is one field describing the baseline decision. The rule uses only observed fields available in the starter dataset.

### Reason codes

The rule will output exactly one reason code for each page:

- **STALE_HIGH_VISIBILITY** — the page is >180 days since its last update and has >=500 impressions in 90 days.
- **STALE_MEDIUM_VISIBILITY** — the page is >180 days since its last update and has 250–499 impressions in 90 days.
- **RECENT_HIGH_VISIBILITY** — the page is not >180 days stale but has >=500 impressions in 90 days.
- **OTHER** — all other combinations.

The reason code is descriptive rather than a second score or label.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the baseline ranked action queue

baseline = df.copy()

# Staleness points: 0–2
baseline["staleness_points"] = baseline["days_since_last_update"].apply(
    lambda x: 2 if x > 180 else (1 if x > 90 else 0)
)

# Search visibility points: 0–2
baseline["visibility_points"] = baseline["impressions_90d"].apply(
    lambda x: 2 if x >= 500 else (1 if x >= 250 else 0)
)

# Final baseline score: 0–4
baseline["score"] = (
    baseline["staleness_points"] +
    baseline["visibility_points"]
)

# Action label
baseline["action"] = baseline["score"].apply(
    lambda x: "Refresh" if x >= 3 else "Monitor"
)

# Exactly one reason code per page
def assign_reason_code(row):
    stale = row["days_since_last_update"] > 180
    high_visibility = row["impressions_90d"] >= 500
    medium_visibility = 250 <= row["impressions_90d"] < 500

    if stale and high_visibility:
        return "STALE_HIGH_VISIBILITY"
    elif stale and medium_visibility:
        return "STALE_MEDIUM_VISIBILITY"
    elif (not stale) and high_visibility:
        return "RECENT_HIGH_VISIBILITY"
    else:
        return "OTHER"

baseline["reason_code"] = baseline.apply(
    assign_reason_code,
    axis=1
)

# Rank highest-priority pages first
baseline = baseline.sort_values(
    by=["score", "impressions_90d", "days_since_last_update"],
    ascending=[False, False, False]
).reset_index(drop=True)

baseline["rank"] = baseline.index + 1

# Keep the queue compact and auditable
queue = baseline[
    [
        "rank",
        "content_id",
        "client_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d"
    ]
].copy()

# Create the required output directory
output_dir = "/content/flyrank-ml-internship/work/outputs"
os.makedirs(output_dir, exist_ok=True)

# Write the required output file
output_path = f"{output_dir}/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print(f"Queue rows: {len(queue):,}")
print(f"Output written to: {output_path}")

print("\nAction counts:")
display(
    queue["action"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="n")
)

print("\nTop 10 baseline queue:")
display(queue.head(10))

Queue rows: 30,000
Output written to: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv

Action counts:


,action,n
0,Monitor,23417
1,Refresh,6583



Top 10 baseline queue:


,rank,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d
0,1,content_cf56e2e2e282,client_7f2253d7e2,4,STALE_HIGH_VISIBILITY,Refresh,194,61678
1,2,content_7368877ea310,client_7f2253d7e2,4,STALE_HIGH_VISIBILITY,Refresh,194,59472
2,3,content_1bfaa38ff26c,client_7f2253d7e2,4,STALE_HIGH_VISIBILITY,Refresh,194,25715
3,4,content_0a91db491d14,client_7f2253d7e2,4,STALE_HIGH_VISIBILITY,Refresh,193,13299
4,5,content_5feee3994adb,client_7f2253d7e2,4,STALE_HIGH_VISIBILITY,Refresh,194,7812
5,6,content_c2d929d83eaa,client_7f2253d7e2,4,STALE_HIGH_VISIBILITY,Refresh,193,7558
6,7,content_b16bd7307b39,client_7f2253d7e2,4,STALE_HIGH_VISIBILITY,Refresh,194,4590
7,8,content_fe16a55cd13d,client_7f2253d7e2,4,STALE_HIGH_VISIBILITY,Refresh,194,4556
8,9,content_ecb6215e79fd,client_7f2253d7e2,4,STALE_HIGH_VISIBILITY,Refresh,194,4429
9,10,content_928af3e22c80,client_7f2253d7e2,4,STALE_HIGH_VISIBILITY,Refresh,193,1697


## 3. Top-10 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Review approach

I reviewed the top ten pages selected by the baseline rule. Each row receives its action and reason code from the rule, followed by a short explanation of why it was selected and what could make the recommendation wrong.

The top ten are all Refresh candidates because they have both high search visibility and more than 180 days since their last update. A limitation of this baseline is that it does not know whether the page is already being intentionally maintained or whether the current traffic is strategically important, so those factors could make a Refresh recommendation wrong.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Review the top 10 baseline recommendations

top10 = queue.head(10).copy()

def review_reason(row):
    return (
        f"Refresh because the page is {int(row['days_since_last_update'])} "
        f"days since its last update and has {int(row['impressions_90d']):,} "
        "impressions in 90 days."
    )

def what_would_make_it_wrong(row):
    return (
        "The recommendation could be wrong if the page is intentionally "
        "left unchanged, is already being maintained, or its current "
        "search visibility is strategically expected."
    )

top10["why_selected"] = top10.apply(review_reason, axis=1)
top10["what_would_make_it_wrong"] = top10.apply(
    what_would_make_it_wrong,
    axis=1
)

top10_review = top10[
    [
        "rank",
        "action",
        "reason_code",
        "why_selected",
        "what_would_make_it_wrong"
    ]
].copy()

print("Top-10 skeptical review:")
display(top10_review)

Top-10 skeptical review:


,rank,action,reason_code,why_selected,what_would_make_it_wrong
0,1,Refresh,STALE_HIGH_VISIBILITY,Refresh because the page is 194 days since its...,The recommendation could be wrong if the page ...
1,2,Refresh,STALE_HIGH_VISIBILITY,Refresh because the page is 194 days since its...,The recommendation could be wrong if the page ...
2,3,Refresh,STALE_HIGH_VISIBILITY,Refresh because the page is 194 days since its...,The recommendation could be wrong if the page ...
3,4,Refresh,STALE_HIGH_VISIBILITY,Refresh because the page is 193 days since its...,The recommendation could be wrong if the page ...
4,5,Refresh,STALE_HIGH_VISIBILITY,Refresh because the page is 194 days since its...,The recommendation could be wrong if the page ...
5,6,Refresh,STALE_HIGH_VISIBILITY,Refresh because the page is 193 days since its...,The recommendation could be wrong if the page ...
6,7,Refresh,STALE_HIGH_VISIBILITY,Refresh because the page is 194 days since its...,The recommendation could be wrong if the page ...
7,8,Refresh,STALE_HIGH_VISIBILITY,Refresh because the page is 194 days since its...,The recommendation could be wrong if the page ...
8,9,Refresh,STALE_HIGH_VISIBILITY,Refresh because the page is 194 days since its...,The recommendation could be wrong if the page ...
9,10,Refresh,STALE_HIGH_VISIBILITY,Refresh because the page is 193 days since its...,The recommendation could be wrong if the page ...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak picks and leakage check

print("Weak-pick review:")
print(
    "The main weak-pick risk is that the top queue is dominated by one client. "
    "The baseline ranks pages independently and does not enforce client-level diversity."
)

# Check for future/label-derived fields in the queue
forbidden_fields = [
    "trend_direction",
    "trend_pct"
]

leaked_fields_present = [
    col for col in forbidden_fields
    if col in queue.columns
]

print("\nLeakage check:")
if leaked_fields_present:
    print("FAIL — forbidden fields found:", leaked_fields_present)
else:
    print(
        "PASS — the ranked queue uses only observed current-data signals "
        "and contains no trend_direction or trend_pct fields."
    )

print("\nQueue columns:")
print(list(queue.columns))

# Show the client concentration in the top 10

print("Top-10 client concentration:")
display(
    top10["client_id"]
    .value_counts()
    .rename_axis("client_id")
    .reset_index(name="top10_count")
)

Weak-pick review:
The main weak-pick risk is that the top queue is dominated by one client. The baseline ranks pages independently and does not enforce client-level diversity.

Leakage check:
PASS — the ranked queue uses only observed current-data signals and contains no trend_direction or trend_pct fields.

Queue columns:
['rank', 'content_id', 'client_id', 'score', 'reason_code', 'action', 'days_since_last_update', 'impressions_90d']
Top-10 client concentration:


,client_id,top10_count
0,client_7f2253d7e2,10


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.